# Fish Speech Sermon Generator (Kaggle Edition)

Optimized for Kaggle P100/T4 GPUs using **fish-speech v1.5.1**.

### Fixes applied vs. original:
- Corrected vocoder module path (`vqgan.inference` not `dac.inference`)
- Fixed broken `codec.pth` symlink (uses absolute paths)
- Fixed `einx` version to be torch-2.4 compatible
- Deferred `faster_whisper` install until after torch pin to prevent re-resolution
- Force-reinstall pinned torch at the end of setup so nothing overwrites it
- Added `codes_0.npy` path discovery step between inference stages
- ffmpeg installed via a static binary to avoid apt mirror hangs


In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────────────────
import os
from pathlib import Path

WORKING_DIR = "/kaggle/working"
os.chdir(WORKING_DIR)

# ── Clone fish-speech ────────────────────────────────────────────────────────
if not Path("fish-speech").exists():
    !git clone https://github.com/fishaudio/fish-speech.git

%cd /kaggle/working/fish-speech
!git checkout v1.5.1

# ── Step 1: Install fish-speech package without pyaudio-blocked deps ─────────
# v1.5.1 depends on pyaudio, which fails to build on Kaggle and is not needed
# for this offline generation path. Install package metadata, then runtime deps.
!pip install -e . --no-deps -q

# Install extra runtime deps.
# NOTE: einx>=0.3.0 is required for torch 2.4 compat (0.2.x breaks).
#       faster_whisper is installed here BEFORE the torch pin so it doesn't
#       drag in a conflicting ctranslate2 torch build afterward.
!pip install -q \
    numpy==1.26.4 \
    "transformers>=4.45.2" datasets==2.18.0 "lightning>=2.1.0" \
    "tensorboard>=2.14.1" "einops>=0.7.0" "librosa>=0.10.1" "rich>=13.5.3" \
    "wandb>=0.15.11" "grpcio>=1.58.0" "uvicorn>=0.30.0" "loguru>=0.6.0" \
    pydantic==2.9.2 \
    vector_quantize_pytorch==1.14.24 \
    "einx[torch]>=0.3.0" \
    hydra-core natsort gradio kui loralib pyrootutils resampy \
    zstandard pydub \
    faster_whisper modelscope==1.17.1 funasr==1.1.5 \
    opencc-python-reimplemented silero-vad \
    ormsgpack cachetools tiktoken \
    huggingface_hub

# ── Step 2: Force-pin torch LAST so nothing above overwrites it ──────────────
# cu121 wheels work on both P100 and T4.
!pip install -q --force-reinstall \
    torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 \
    --index-url https://download.pytorch.org/whl/cu121
!pip install -q --force-reinstall numpy==1.26.4

# ── Step 3: Install ffmpeg via static binary (avoids apt mirror hangs) ───────
!pip install -q ffmpeg-python
import shutil
if not shutil.which("ffmpeg"):
    # Kaggle images usually ship ffmpeg; if not, pull a static build.
    !wget -q https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz -O /tmp/ffmpeg.tar.xz
    !tar -xf /tmp/ffmpeg.tar.xz -C /tmp/
    !cp /tmp/ffmpeg-*-amd64-static/ffmpeg /usr/local/bin/ffmpeg
    !chmod +x /usr/local/bin/ffmpeg
    print("ffmpeg installed from static binary.")
else:
    print(f"ffmpeg already available at: {shutil.which('ffmpeg')}")

print("\n✅ Environment setup complete.")


In [ ]:
# ── Cell 2: Verify GPU & torch ───────────────────────────────────────────────
import torch
print(f"torch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── Cell 3: Download Model Weights ───────────────────────────────────────────
# Downloads to checkpoints/fish-speech-1.5 inside the repo dir.
%cd /kaggle/working/fish-speech

CHECKPOINT_DIR = "checkpoints/fish-speech-1.5"
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="fishaudio/fish-speech-1.5",
    local_dir=CHECKPOINT_DIR,
)

# ── Create codec.pth symlink with ABSOLUTE paths to avoid broken links ────────
import os
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/fish-speech")
VOCODER_SRC = (REPO_ROOT / CHECKPOINT_DIR / "firefly-gan-vq-fsq-8x1024-21hz-generator.pth").resolve()
CODEC_LINK   = (REPO_ROOT / CHECKPOINT_DIR / "codec.pth").resolve()

if CODEC_LINK.exists() or CODEC_LINK.is_symlink():
    CODEC_LINK.unlink()

if VOCODER_SRC.exists():
    os.symlink(VOCODER_SRC, CODEC_LINK)
    print(f"✅ Symlink created: {CODEC_LINK} → {VOCODER_SRC}")
else:
    print(f"⚠️  Vocoder checkpoint not found at {VOCODER_SRC}")
    print("   Listing checkpoint dir:")
    for f in (REPO_ROOT / CHECKPOINT_DIR).iterdir():
        print(f"   {f.name}")


In [ ]:
# ── Cell 4: Prepare Seed Audio ───────────────────────────────────────────────
import glob
import os

SEARCH_PATTERNS = [
    "/kaggle/input/**/spurgeon_seed.wav",
    "/kaggle/working/spurgeon_seed.wav",
    "/kaggle/working/fish-speech/spurgeon_seed.wav",
]

REFERENCE_AUDIO = None
for pattern in SEARCH_PATTERNS:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        REFERENCE_AUDIO = matches[0]
        print(f"✅ Found seed audio at: {REFERENCE_AUDIO}")
        break

if not REFERENCE_AUDIO:
    raise FileNotFoundError(
        "⚠️  Seed audio (spurgeon_seed.wav) not found!\n"
        "    Attach a Kaggle dataset containing the file, or upload it to /kaggle/working/."
    )

# Load matching .lab transcript if present
PROMPT_TEXT_PATH = REFERENCE_AUDIO.replace(".wav", ".lab")
if os.path.exists(PROMPT_TEXT_PATH):
    with open(PROMPT_TEXT_PATH, "r") as f:
        prompt_text = f.read().strip()
    print(f"✅ Loaded prompt text from: {PROMPT_TEXT_PATH}")
else:
    prompt_text = "Welcome to the Metropolitan Tabernacle."
    print("⚠️  No .lab file found — using default prompt text.")

print(f"\nPrompt text preview: {prompt_text[:80]}...")


In [ ]:
# ── Cell 5: Generate Sermon Audio ────────────────────────────────────────────
import subprocess
import glob
from pathlib import Path

%cd /kaggle/working/fish-speech

sermon_script = """
[solemn]
"NOT UNTO US, O LORD, NOT UNTO US, BUT UNTO THY NAME GIVE GLORY, FOR THY MERCY, AND FOR THY TRUTH'S SAKE." [pause] — PSALM 115, verse 1.

[warm]
Every careful reader can see the connection between this 115th Psalm and the one which precedes it. In the 114th Psalm, we see the gracious and grateful Jews sitting around the passover table, having eaten of the lamb, and singing of the miracles of Jehovah at the Red Sea and the Jordan.

[excited, narrative weight]
It must have been a very jubilant song that they sang; I think I can hear them singing, [emphasis] "What ailed thee, O thou sea, that thou fleddest? thou Jordan, that thou wast driven back?" [pause]

[solemn]
When that joyful hymn was finished, and the cup of wine was passed round the table, they struck another note. They remembered their sad condition, as they heard the heathen say, [whisper] "Where is now their God?"

[assertive]
They recollected that, perhaps for many a year, there had been no miracle, no prophet, no open vision, and then they began to chant a prayer that God would appear—not for their sakes, but for his own name's sake. [pause]

[solemn]
The psalmist's prayer here is, practically, "Lord, do the like again;—not for our sakes, but for thine own name's sake;—that once again the heathen all around may know that there is a God in the midst of Israel." [pause]

[solemn, deep]
First, they furnish us with A POWERFUL PLEA IN PRAYER: "Not unto us, O Jehovah, not unto us, but unto thy name give glory."

[compassionate]
There are times when this is the only plea that God's people can use. There are other occasions when we can plead with God to bless us, for this reason or for that; but, sometimes, there come [emphasis] dark experiences, when there seems to be no reason that can suggest itself to us why God should give us deliverance, except this one—that he would be pleased to do it in order to [emphasis] glorify his own name. [pause]

[narrative, warm]
Moses is an example of how this plea prevails with the Lord. When he was on the mount with God, and Jehovah threatened to destroy the idolatrous Israelites, Moses pleaded: [solemn] "Wherefore should the Egyptians speak, and say, For mischief did he bring them out, to slay them in the mountains?" [pause]

[assertive]
And the Lord repented of the evil which he thought to do unto his people. [pause] Joshua also used the same plea when he said to the Lord, after Israel's defeat at Ai, [whisper, intensity] "What wilt thou do unto thy great name?" [pause]

[warm, compassionate]
Let me just whisper a word in the ear of anyone who has scarcely learnt to pray. Poor sinner, [pause] laden with guilt and full of fears, [pause] thou sayest, "How can I plead with God for mercy? I have rejected it for years."

[solemn, encouraging]
Here is one for thee to use; say to him, "For thy mercy and thy love's sake, have pity upon me, the least deserving of all thy creatures; [pause] for, surely, if thou wilt but save me, it will be an eternal wonder to men and to angels." [pause]

[assertive]
Now, secondly, my text appears to me to embody THE TRUE SPIRIT OF PIETY. [pause] That is to say, true religion does not seek its own honour. [pause]

[solemn]
Self-seeking is the exact opposite of the spirit of a true Christian. He would rather strip himself, and say, [emphasis] "Not unto me, but unto thee, O Lord, be all honour and glory!" [pause]

[narrative, slightly amused]
Somebody once told Master John Bunyan that he had preached a delightful sermon. [pause] [whisper] "You are too late," said John, "the devil told me that before I left the pulpit." [pause]

[assertive]
Satan is a great adept in teaching us how to steal our Master's glory. Yet, if ever we speak aright, it is because we are taught of the Spirit, and not because of our own wisdom. [pause]

[solemn]
Finally, my text seems to contain within itself THE ACCEPTABLE SPIRIT IN WHICH TO REVIEW THE PAST. [pause]

[warm, encouraging]
Has God blessed us? Do we look back upon honourable and useful lives? Then, let us be sure to stick to the text: "Not unto us, O Lord, not unto us." [pause]

[assertive, solemn]
Now, young man, if you are beginning to serve the Saviour, and he has given you success, [pause] your conduct in this first time of testing may decide the whole of your future life. [pause] He is not going to put his treasure into the leaky vessel of self-exaltation. [pause]

[solemn, hopeful]
Ay, and when the time comes for us to die, this is the spirit in which to die, for it is the beginning of heaven. [pause] What are they doing in heaven? They will not wear their crowns; no, not they; but they cast them down at Christ's feet, crying, [emphasis] "Not unto us, O Lord, not unto us, but unto thy name give glory." [pause]

[warm, compassionate]
Recollect that the end of the creature is the beginning of the Creator. [pause] When you have done with every other confidence, then you can have confidence in God. [pause]

[solemn]
The Lord bless you to this end, [pause] for Jesus Christ's sake! [pause] Amen.
"""

CHECKPOINT_DIR = "checkpoints/fish-speech-1.5"
VOCODER_CKPT = f"{CHECKPOINT_DIR}/firefly-gan-vq-fsq-8x1024-21hz-generator.pth"
PROMPT_TOKENS_PATH = "prompt_tokens.npy"

# ── Stage 0: Reference Audio → Prompt Tokens ────────────────────────────────
print("Stage 0: Reference audio → prompt tokens...")
subprocess.run(
    [
        "python", "-m", "fish_speech.models.vqgan.inference",
        "-i", REFERENCE_AUDIO,
        "-o", "prompt_tokens.wav",
        "--checkpoint-path", VOCODER_CKPT,
    ],
    check=True,
    capture_output=False,
)
if not Path(PROMPT_TOKENS_PATH).exists():
    raise FileNotFoundError(f"Prompt tokens not created: {PROMPT_TOKENS_PATH}")

# ── Stage 1: Text → Semantic Tokens ─────────────────────────────────────────
print("Stage 1: Text → Semantic tokens...")
result = subprocess.run(
    [
        "python", "-m", "fish_speech.models.text2semantic.inference",
        "--text", sermon_script,
        "--prompt-text", prompt_text,
        "--prompt-tokens", PROMPT_TOKENS_PATH,
        "--checkpoint-path", CHECKPOINT_DIR,
        "--num-samples", "1",
        "--half",
    ],
    check=True,
    capture_output=False,
)

# ── Locate the generated codes file ─────────────────────────────────────────
# Output location varies slightly by version; search to be safe.
codes_candidates = glob.glob("**/codes_0.npy", recursive=True) + glob.glob("codes_0.npy")
if not codes_candidates:
    raise FileNotFoundError(
        "codes_0.npy not found after Stage 1.\n"
        "Check Stage 1 output above for errors or an alternate filename."
    )
CODES_PATH = codes_candidates[0]
print(f"✅ codes_0.npy found at: {CODES_PATH}")

# ── Stage 2: Semantic Tokens → Audio ────────────────────────────────────────
# FIX: correct module path is fish_speech.models.vqgan.inference
#      (NOT fish_speech.models.dac.inference — that module does not exist)
print("\nStage 2: Semantic tokens → audio...")
subprocess.run(
    [
        "python", "-m", "fish_speech.models.vqgan.inference",
        "-i", CODES_PATH,
        "--checkpoint-path", VOCODER_CKPT,
    ],
    check=True,
    capture_output=False,
)

print("\n✅ Generation complete! Output saved as fake.wav")


In [ ]:
# ── Cell 6: Collect Output & Optional Webhook Notification ───────────────────
import shutil
import glob
from pathlib import Path

# fish-speech writes fake.wav into the repo root by default
OUTPUT_CANDIDATES = glob.glob("/kaggle/working/fish-speech/fake.wav") + glob.glob("fake.wav")

if not OUTPUT_CANDIDATES:
    raise FileNotFoundError("fake.wav not found — check Stage 2 output above.")

SRC  = OUTPUT_CANDIDATES[0]
DEST = "/kaggle/working/sermon_output.wav"
shutil.copy(SRC, DEST)
print(f"✅ Copied {SRC} → {DEST}")

# ── Optional: play inline in the notebook ────────────────────────────────────
from IPython.display import Audio, display
display(Audio(DEST))

# ── Optional: webhook notification ───────────────────────────────────────────
WEBHOOK_URL = ""  # PASTE YOUR WEBHOOK URL HERE
if WEBHOOK_URL:
    import requests
    resp = requests.post(WEBHOOK_URL, json={"status": "complete", "file": "sermon_output.wav"})
    print(f"Webhook response: {resp.status_code}")
